# FSDP (Fully Sharded Data Parallel) - PyTorch Native

**Historical Context**: FSDP was introduced by Facebook/Meta AI in PyTorch 1.11 (2021) as a native PyTorch implementation of the ZeRO-3 concept. It's now the recommended way to train large models in PyTorch without external dependencies.

**Key Innovation**: FSDP brings ZeRO-style sharding directly into PyTorch with:
- Native PyTorch API (no external libraries needed)
- Automatic mixed precision integration
- Flexible sharding strategies
- Better composability with PyTorch ecosystem

**Learning Objectives**:
- Understand FSDP fundamentals (PyTorch 2021)
- Compare FSDP with DeepSpeed ZeRO
- Master different sharding strategies
- Implement mixed precision with FSDP
- Learn transformer wrapping policies
- Set up multi-GPU training
- Compare FSDP vs DDP performance

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Optional
import pandas as pd

# FSDP imports (available in PyTorch >= 1.11)
try:
    from torch.distributed.fsdp import (
        FullyShardedDataParallel as FSDP,
        MixedPrecision,
        BackwardPrefetch,
        ShardingStrategy,
        CPUOffload,
    )
    from torch.distributed.fsdp.wrap import (
        size_based_auto_wrap_policy,
        enable_wrap,
        wrap,
    )
    FSDP_AVAILABLE = True
    print(f"PyTorch version: {torch.__version__}")
    print("FSDP is available!")
except ImportError:
    FSDP_AVAILABLE = False
    print(f"PyTorch version: {torch.__version__}")
    print("FSDP not available. Requires PyTorch >= 1.11")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

## 1. FSDP Fundamentals

**How FSDP Works**:

FSDP wraps your model (or submodules) and automatically:
1. **Shards** parameters, gradients, and optimizer states across GPUs
2. **All-Gathers** full parameters before forward/backward
3. **Discards** non-owned parameters after computation
4. **Reduces** gradients across ranks

**Key Concepts**:

- **Flat Parameter**: FSDP flattens module parameters into a single tensor
- **Sharding**: Each rank owns 1/N of parameters
- **All-Gather**: Temporary collection of full parameters
- **Resharding**: Release parameters after use

**Comparison with DDP**:
```
DDP (DistributedDataParallel):
- Each GPU: Full model copy
- Memory: O(model_size) per GPU
- Communication: AllReduce gradients

FSDP (Fully Sharded Data Parallel):
- Each GPU: 1/N of model
- Memory: O(model_size/N) per GPU
- Communication: AllGather params + AllReduce gradients
```

In [ ]:
# Visualize FSDP vs DDP
def visualize_fsdp_vs_ddp():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # DDP
    ax = axes[0]
    num_gpus = 4
    model_parts = ['Params', 'Gradients', 'Optimizer']
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    for gpu in range(num_gpus):
        bottom = 0
        for part, color in zip(model_parts, colors):
            ax.bar(gpu, 1, bottom=bottom, color=color, alpha=0.7, 
                  edgecolor='black', linewidth=2, width=0.8)
            if gpu == 0:
                ax.text(-0.8, bottom + 0.5, part, fontsize=10, va='center')
            bottom += 1
    
    ax.set_xlim(-1, num_gpus)
    ax.set_ylim(0, 3.5)
    ax.set_xlabel('GPU Rank', fontsize=12)
    ax.set_ylabel('Model Components', fontsize=12)
    ax.set_title('DDP: Full Replication\n(Each GPU has complete copy)', 
                fontsize=12, fontweight='bold')
    ax.set_xticks(range(num_gpus))
    ax.set_yticks([])
    ax.text(num_gpus/2 - 0.5, 3.2, 'Memory: O(M) per GPU', 
           fontsize=11, fontweight='bold', ha='center',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # FSDP
    ax = axes[1]
    colors_fsdp = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    for gpu in range(num_gpus):
        bottom = 0
        for idx, (part, color) in enumerate(zip(model_parts, colors_fsdp)):
            # Each GPU only has a shard
            ax.bar(gpu, 1, bottom=bottom, color=color, alpha=0.7,
                  edgecolor='black', linewidth=2, width=0.8)
            # Add shard number
            ax.text(gpu, bottom + 0.5, f'Shard\n{gpu}', 
                   ha='center', va='center', fontsize=9, fontweight='bold')
            if gpu == 0:
                ax.text(-0.8, bottom + 0.5, part, fontsize=10, va='center')
            bottom += 1
    
    ax.set_xlim(-1, num_gpus)
    ax.set_ylim(0, 3.5)
    ax.set_xlabel('GPU Rank', fontsize=12)
    ax.set_ylabel('Model Components', fontsize=12)
    ax.set_title('FSDP: Sharded Across GPUs\n(Each GPU has 1/N of model)', 
                fontsize=12, fontweight='bold')
    ax.set_xticks(range(num_gpus))
    ax.set_yticks([])
    ax.text(num_gpus/2 - 0.5, 3.2, 'Memory: O(M/N) per GPU', 
           fontsize=11, fontweight='bold', ha='center',
           bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/fsdp_vs_ddp.png',
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_fsdp_vs_ddp()

## 2. FSDP vs DeepSpeed ZeRO Comparison

**Similarities**:
- Both implement parameter/gradient/optimizer sharding
- Both reduce memory redundancy
- Both use AllGather for parameter collection

**Key Differences**:

In [ ]:
comparison_data = {
    'Aspect': [
        'Integration',
        'API Style',
        'Configuration',
        'Sharding Flexibility',
        'Mixed Precision',
        'CPU Offloading',
        'Activation Checkpointing',
        'Model Compatibility',
        'Learning Curve',
        'Maturity',
        'Best For',
    ],
    'FSDP (PyTorch Native)': [
        'Native PyTorch (no deps)',
        'Pythonic, wrapping-based',
        'Python code',
        'Module-level control',
        'Native torch.amp',
        'Yes (simple)',
        'Native PyTorch',
        'Any PyTorch model',
        'Easier (familiar API)',
        'Stable since 2021',
        'PyTorch users, research',
    ],
    'DeepSpeed ZeRO': [
        'External library',
        'Engine-based',
        'JSON config file',
        'Stage-based (1/2/3)',
        'Custom FP16 engine',
        'Yes (advanced, NVMe)',
        'DeepSpeed API',
        'Requires adaptation',
        'Steeper (new concepts)',
        'Very mature (2020+)',
        'Production, very large models',
    ],
}

df = pd.DataFrame(comparison_data)
print("FSDP vs DeepSpeed ZeRO Comparison:")
print("="*100)
print(df.to_string(index=False))
print("="*100)

print("\nPerformance Comparison:")
print("-"*100)
print("Speed: Generally similar for Stage 3 / FULL_SHARD")
print("Memory: Nearly identical memory savings")
print("Ease of Use: FSDP easier for PyTorch users")
print("Advanced Features: DeepSpeed has more (ZeRO++, Infinity, etc.)")
print("Ecosystem: FSDP better integrated with PyTorch tools")

## 3. FSDP Sharding Strategies

FSDP offers several sharding strategies:

### ShardingStrategy Options:

1. **FULL_SHARD** (Default, equivalent to ZeRO-3):
   - Shards parameters, gradients, optimizer states
   - Maximum memory savings
   - More communication

2. **SHARD_GRAD_OP** (equivalent to ZeRO-2):
   - Shards gradients and optimizer states only
   - Parameters are replicated
   - Less communication

3. **NO_SHARD** (equivalent to DDP):
   - No sharding, full replication
   - Useful for debugging
   - Same as DistributedDataParallel

4. **HYBRID_SHARD** (Hierarchical):
   - Shards within nodes, replicates across nodes
   - Good for multi-node training
   - Balances intra-node and inter-node communication

5. **_HYBRID_SHARD_ZERO2**:
   - Hybrid with ZeRO-2 style
   - Node-local ZeRO-2

In [ ]:
# Visualize sharding strategies
def visualize_sharding_strategies():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    strategies = [
        {
            'name': 'FULL_SHARD\n(ZeRO-3 style)',
            'params': [0.25, 0.25, 0.25, 0.25],
            'grads': [0.25, 0.25, 0.25, 0.25],
            'opt': [0.25, 0.25, 0.25, 0.25],
        },
        {
            'name': 'SHARD_GRAD_OP\n(ZeRO-2 style)',
            'params': [1.0, 1.0, 1.0, 1.0],
            'grads': [0.25, 0.25, 0.25, 0.25],
            'opt': [0.25, 0.25, 0.25, 0.25],
        },
        {
            'name': 'NO_SHARD\n(DDP style)',
            'params': [1.0, 1.0, 1.0, 1.0],
            'grads': [1.0, 1.0, 1.0, 1.0],
            'opt': [1.0, 1.0, 1.0, 1.0],
        },
        {
            'name': 'HYBRID_SHARD\n(2 nodes, 2 GPUs each)',
            'params': [0.5, 0.5, 0.5, 0.5],  # Sharded within node
            'grads': [0.5, 0.5, 0.5, 0.5],
            'opt': [0.5, 0.5, 0.5, 0.5],
        },
    ]
    
    for idx, (ax, strategy) in enumerate(zip(axes, strategies)):
        x = np.arange(4)  # 4 GPUs
        width = 0.25
        
        bars1 = ax.bar(x - width, strategy['params'], width, 
                      label='Parameters', alpha=0.8, color='#FF6B6B')
        bars2 = ax.bar(x, strategy['grads'], width,
                      label='Gradients', alpha=0.8, color='#4ECDC4')
        bars3 = ax.bar(x + width, strategy['opt'], width,
                      label='Optimizer', alpha=0.8, color='#45B7D1')
        
        ax.set_ylabel('Fraction of Full Size', fontsize=11)
        ax.set_xlabel('GPU Rank', fontsize=11)
        ax.set_title(strategy['name'], fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([f'GPU {i}' for i in range(4)])
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, 1.2)
        
        # Calculate total memory
        total_mem = strategy['params'][0] + strategy['grads'][0] + strategy['opt'][0]
        ax.text(1.5, 1.1, f'Memory per GPU: {total_mem:.2f}x', 
               ha='center', fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
        
        # Add node boundaries for hybrid
        if 'HYBRID' in strategy['name']:
            ax.axvline(x=1.5, color='red', linestyle='--', linewidth=2, alpha=0.5)
            ax.text(0.5, 1.05, 'Node 0', ha='center', fontsize=9, color='red', fontweight='bold')
            ax.text(2.5, 1.05, 'Node 1', ha='center', fontsize=9, color='red', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/fsdp_sharding_strategies.png',
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_sharding_strategies()

# Memory comparison table
print("\nMemory Usage by Sharding Strategy (6.7B parameter model, 4 GPUs):")
print("="*80)

num_params = 6_700_000_000
num_gpus = 4

strategies_mem = [
    ('FULL_SHARD', 1/num_gpus, 1/num_gpus, 1/num_gpus),
    ('SHARD_GRAD_OP', 1.0, 1/num_gpus, 1/num_gpus),
    ('NO_SHARD', 1.0, 1.0, 1.0),
]

data = []
for name, p_factor, g_factor, o_factor in strategies_mem:
    params_gb = (num_params * 6 * p_factor) / (1024**3)  # FP16+FP32
    grads_gb = (num_params * 2 * g_factor) / (1024**3)
    opt_gb = (num_params * 8 * o_factor) / (1024**3)
    total_gb = params_gb + grads_gb + opt_gb
    
    data.append({
        'Strategy': name,
        'Params (GB)': f'{params_gb:.2f}',
        'Grads (GB)': f'{grads_gb:.2f}',
        'Optimizer (GB)': f'{opt_gb:.2f}',
        'Total (GB)': f'{total_gb:.2f}',
    })

df = pd.DataFrame(data)
print(df.to_string(index=False))
print("="*80)

## 4. Mixed Precision with FSDP

FSDP integrates seamlessly with PyTorch's native mixed precision training:

In [ ]:
if FSDP_AVAILABLE:
    # Example mixed precision configurations
    
    # 1. BFloat16 mixed precision (recommended for A100/H100)
    bfloat16_policy = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )
    
    # 2. FP16 mixed precision
    fp16_policy = MixedPrecision(
        param_dtype=torch.float16,
        reduce_dtype=torch.float16,
        buffer_dtype=torch.float16,
    )
    
    # 3. Pure FP32 (for comparison/debugging)
    fp32_policy = MixedPrecision(
        param_dtype=torch.float32,
        reduce_dtype=torch.float32,
        buffer_dtype=torch.float32,
    )
    
    print("FSDP Mixed Precision Configurations:")
    print("="*70)
    print("\n1. BFloat16 (Recommended):")
    print("   - Best for A100/H100 GPUs")
    print("   - Better numerical stability than FP16")
    print("   - Same range as FP32")
    print("   - 2x memory savings")
    
    print("\n2. FP16:")
    print("   - Works on most GPUs")
    print("   - May need gradient scaling")
    print("   - 2x memory savings")
    print("   - Slightly less stable than BF16")
    
    print("\n3. FP32:")
    print("   - Full precision")
    print("   - Debugging and verification")
    print("   - No memory savings")
    print("="*70)
else:
    print("FSDP not available. Install PyTorch >= 1.11")

# Precision comparison table
precision_data = {
    'Precision': ['FP32', 'FP16', 'BFloat16'],
    'Bits': [32, 16, 16],
    'Range': ['±3.4e38', '±6.5e4', '±3.4e38'],
    'Precision': ['7 decimal', '3 decimal', '2-3 decimal'],
    'Memory': ['1x', '0.5x', '0.5x'],
    'Speed': ['1x', '2-4x', '2-4x'],
    'Stability': ['Best', 'Needs care', 'Good'],
    'GPU Support': ['All', 'V100+', 'A100+'],
}

df = pd.DataFrame(precision_data)
print("\nPrecision Comparison:")
print("="*100)
print(df.to_string(index=False))
print("="*100)

## 5. Transformer Wrapping Policy

**Wrapping Policy** determines which modules get wrapped by FSDP:

- **Layer-wise wrapping**: Each transformer layer is its own FSDP unit
- **Size-based wrapping**: Wrap based on parameter count
- **Manual wrapping**: Explicit control over wrapping

**Why it matters**:
- Affects communication granularity
- Impacts memory-communication trade-off
- Determines resharding frequency

In [ ]:
# Example transformer model
class TransformerBlock(nn.Module):
    def __init__(self, d_model=512, nhead=8):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Self-attention with residual
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        
        # FFN with residual
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x

class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, nhead) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.fc_out(x)
        return x

# Wrapping policy examples
if FSDP_AVAILABLE:
    wrapping_examples = '''
# 1. Size-based auto wrapping (wrap modules with > 100M parameters)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy

my_auto_wrap_policy = functools.partial(
    size_based_auto_wrap_policy, 
    min_num_params=100_000_000
)

model = FSDP(
    model,
    auto_wrap_policy=my_auto_wrap_policy,
)

# 2. Layer-based wrapping (wrap each TransformerBlock)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

my_auto_wrap_policy = functools.partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={
        TransformerBlock,
    },
)

model = FSDP(
    model,
    auto_wrap_policy=my_auto_wrap_policy,
)

# 3. Manual wrapping
model = SimpleTransformer()

# Wrap each layer individually
for i, layer in enumerate(model.layers):
    model.layers[i] = FSDP(layer)

# Wrap entire model
model = FSDP(model)

# 4. Hybrid wrapping
from torch.distributed.fsdp.wrap import enable_wrap, wrap

with enable_wrap(wrapper_cls=FSDP):
    model = SimpleTransformer()
    # Selectively wrap layers
    for i in range(len(model.layers)):
        model.layers[i] = wrap(model.layers[i])
    model = wrap(model)
'''
    print("FSDP Wrapping Policy Examples:")
    print("="*70)
    print(wrapping_examples)

# Visualize wrapping granularity
def visualize_wrapping_granularity():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    scenarios = [
        ('No Wrapping\n(Single FSDP unit)', 1, ['Entire Model']),
        ('Layer Wrapping\n(6 FSDP units)', 6, [f'Layer {i}' for i in range(6)]),
        ('Fine-grained\n(18 FSDP units)', 18, [f'Block {i}' for i in range(18)]),
    ]
    
    for ax, (title, num_units, labels) in zip(axes, scenarios):
        colors = plt.cm.tab20(np.linspace(0, 1, num_units))
        
        for i in range(num_units):
            ax.barh(0, 1/num_units, left=i/num_units, color=colors[i], 
                   edgecolor='black', linewidth=2)
        
        ax.set_xlim(0, 1)
        ax.set_ylim(-0.5, 0.5)
        ax.set_yticks([])
        ax.set_xticks([])
        ax.set_title(title, fontsize=12, fontweight='bold')
        
        # Add annotations
        ax.text(0.5, -0.35, f'{num_units} FSDP units', ha='center', fontsize=10)
        
        # Communication frequency
        if num_units == 1:
            comm_text = 'Least communication\nMost memory'
            color = 'lightcoral'
        elif num_units == 6:
            comm_text = 'Balanced\n(Recommended)'
            color = 'lightgreen'
        else:
            comm_text = 'Most communication\nLeast memory'
            color = 'lightyellow'
        
        ax.text(0.5, 0.35, comm_text, ha='center', fontsize=9,
               bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/fsdp_wrapping_granularity.png',
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_wrapping_granularity()

## 6. Multi-GPU Training Setup

Complete example of FSDP multi-GPU training:

In [ ]:
# Complete FSDP training script
fsdp_training_script = '''
# train_fsdp.py
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    BackwardPrefetch,
    ShardingStrategy,
    CPUOffload,
)
from torch.distributed.fsdp.wrap import (
    size_based_auto_wrap_policy,
    transformer_auto_wrap_policy,
)
from torch.utils.data import DataLoader, DistributedSampler
import functools

def setup(rank, world_size):
    """Initialize distributed training."""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup():
    """Clean up distributed training."""
    dist.destroy_process_group()

def train(rank, world_size):
    """Main training function."""
    setup(rank, world_size)
    
    # Create model
    model = SimpleTransformer().to(rank)
    
    # Configure FSDP
    # 1. Mixed precision
    mixed_precision_policy = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )
    
    # 2. Wrapping policy
    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={TransformerBlock},
    )
    
    # 3. Wrap model with FSDP
    model = FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        mixed_precision=mixed_precision_policy,
        auto_wrap_policy=auto_wrap_policy,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
        device_id=torch.cuda.current_device(),
    )
    
    # Create optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    
    # Create dataset and dataloader
    dataset = DummyDataset()
    sampler = DistributedSampler(
        dataset, 
        num_replicas=world_size, 
        rank=rank,
        shuffle=True
    )
    dataloader = DataLoader(
        dataset, 
        batch_size=8, 
        sampler=sampler,
        num_workers=2,
        pin_memory=True
    )
    
    # Training loop
    model.train()
    for epoch in range(3):
        sampler.set_epoch(epoch)
        
        for batch_idx, batch in enumerate(dataloader):
            batch = batch.to(rank)
            
            # Forward pass
            outputs = model(batch)
            loss = outputs.mean()
            
            # Backward pass
            loss.backward()
            
            # Optimizer step
            optimizer.step()
            optimizer.zero_grad()
            
            if rank == 0 and batch_idx % 10 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")
    
    # Save checkpoint (only rank 0)
    if rank == 0:
        # Save full state dict
        from torch.distributed.fsdp import FullStateDictConfig, StateDictType
        
        save_policy = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
        with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_policy):
            state_dict = model.state_dict()
            torch.save(state_dict, "model_checkpoint.pt")
    
    cleanup()

if __name__ == "__main__":
    import torch.multiprocessing as mp
    
    world_size = torch.cuda.device_count()
    mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)

# Run with:
# python train_fsdp.py
# Or with torchrun:
# torchrun --nproc_per_node=4 train_fsdp.py
'''

print("Complete FSDP Training Script:")
print("="*70)
print(fsdp_training_script)

# Save the script
with open('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/train_fsdp.py', 'w') as f:
    f.write(fsdp_training_script)
print("\nSaved: train_fsdp.py")

## 7. FSDP vs DDP Performance Comparison

Let's compare memory and speed characteristics:

In [ ]:
def compare_fsdp_ddp():
    """
    Compare FSDP and DDP across different model sizes and GPU counts.
    """
    model_sizes = [1.3e9, 6.7e9, 13e9]  # 1.3B, 6.7B, 13B
    model_names = ['1.3B', '6.7B', '13B']
    gpu_counts = [1, 2, 4, 8]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Memory comparison
    ax = axes[0]
    for model_size, model_name in zip(model_sizes, model_names):
        ddp_memory = []
        fsdp_memory = []
        
        for num_gpus in gpu_counts:
            # DDP: same memory on each GPU
            ddp_mem = (model_size * 16) / (1024**3)  # 16 bytes per param
            ddp_memory.append(ddp_mem)
            
            # FSDP: divided by number of GPUs
            fsdp_mem = (model_size * 16) / (num_gpus * 1024**3)
            fsdp_memory.append(fsdp_mem)
        
        ax.plot(gpu_counts, ddp_memory, 'o--', label=f'DDP ({model_name})', linewidth=2, markersize=8, alpha=0.6)
        ax.plot(gpu_counts, fsdp_memory, 's-', label=f'FSDP ({model_name})', linewidth=2, markersize=8)
    
    ax.axhline(y=40, color='red', linestyle='--', linewidth=2, label='A100 40GB', alpha=0.5)
    ax.axhline(y=80, color='orange', linestyle='--', linewidth=2, label='A100 80GB', alpha=0.5)
    
    ax.set_xlabel('Number of GPUs', fontsize=12)
    ax.set_ylabel('Memory per GPU (GB)', fontsize=12)
    ax.set_title('Memory Usage: FSDP vs DDP', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    ax.set_xticks(gpu_counts)
    ax.set_yscale('log')
    
    # Communication overhead comparison
    ax = axes[1]
    
    x = np.arange(len(gpu_counts))
    width = 0.35
    
    # Relative communication volume (normalized)
    ddp_comm = [1.0, 1.0, 1.0, 1.0]  # AllReduce: constant comm
    fsdp_comm = [1.0, 1.2, 1.5, 1.8]  # AllGather + AllReduce: increases slightly
    
    bars1 = ax.bar(x - width/2, ddp_comm, width, label='DDP', alpha=0.8, color='#FF6B6B')
    bars2 = ax.bar(x + width/2, fsdp_comm, width, label='FSDP', alpha=0.8, color='#4ECDC4')
    
    ax.set_xlabel('Number of GPUs', fontsize=12)
    ax.set_ylabel('Relative Communication Volume', fontsize=12)
    ax.set_title('Communication Overhead', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(gpu_counts)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/fsdp_vs_ddp_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()

compare_fsdp_ddp()

# Detailed comparison table
comparison_table = {
    'Aspect': [
        'Memory per GPU',
        'Max Model Size',
        'Communication',
        'Speed (small models)',
        'Speed (large models)',
        'Setup Complexity',
        'Flexibility',
        'Best For',
    ],
    'DDP': [
        'O(M) - Full copy',
        'Limited by 1 GPU',
        'AllReduce gradients',
        'Fastest',
        'OOM on large models',
        'Simple',
        'Basic',
        'Small-medium models',
    ],
    'FSDP': [
        'O(M/N) - Sharded',
        'Scales with GPUs',
        'AllGather + AllReduce',
        'Slightly slower',
        'Enables training',
        'Moderate',
        'Highly flexible',
        'Large models (>1B params)',
    ],
}

df = pd.DataFrame(comparison_table)
print("\n\nFSDP vs DDP: Detailed Comparison")
print("="*100)
print(df.to_string(index=False))
print("="*100)

print("\nRule of Thumb:")
print("-"*100)
print("Use DDP when: Model fits comfortably on 1 GPU")
print("Use FSDP when: Model is >1B parameters or tight on GPU memory")
print("FSDP overhead: ~10-20% slower than DDP for small models, enables large models")

## Summary

**Key Takeaways**:

1. **FSDP**: PyTorch's native implementation of fully sharded training (ZeRO-3 style)

2. **Advantages over DeepSpeed**:
   - Native PyTorch (no external dependencies)
   - Pythonic API
   - Better ecosystem integration
   - Easier to use and debug

3. **Sharding Strategies**:
   - FULL_SHARD: Maximum memory savings (ZeRO-3)
   - SHARD_GRAD_OP: Balance speed and memory (ZeRO-2)
   - HYBRID_SHARD: Optimized for multi-node
   - NO_SHARD: Same as DDP (debugging)

4. **Mixed Precision**:
   - BFloat16 recommended for A100/H100
   - FP16 for older GPUs
   - Native integration with torch.amp

5. **Wrapping Policy**:
   - Layer-wise wrapping recommended for transformers
   - Affects communication granularity
   - Balance between memory and speed

6. **When to Use**:
   - Models > 1B parameters
   - Memory-constrained scenarios
   - PyTorch-native workflow preferred
   - Research and development

7. **FSDP vs DDP**:
   - FSDP: Better memory, enables large models
   - DDP: Faster for small models
   - Trade ~10-20% speed for 4-8x memory

**Papers & Resources**:
- FSDP Paper: [PyTorch FSDP: Experiences on Scaling Fully Sharded Data Parallel](https://arxiv.org/abs/2304.11277) (Zhao et al., 2023)
- PyTorch Docs: https://pytorch.org/docs/stable/fsdp.html
- Based on ZeRO: [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054) (Rajbhandari et al., 2020)